# 02 — Train YOLOv11

Trains YOLOv11-Medium on the BDD100K clear-weather training split.
Checkpoints are saved every 10 epochs to `checkpoints/yolov11/`.

**Prerequisites:** run `01_setup_and_data.ipynb` first.

In [ ]:
from pathlib import Path
import torch
from dotenv import load_dotenv
from ultralytics import YOLO
from src.train_utils import setup_logging

DRIVE_ROOT  = Path('/content/drive/MyDrive/FON/master_rad/')
CONFIGS_DIR = Path('/content/data_prepared/configs')
CHECKPOINTS = DRIVE_ROOT / 'checkpoints'

load_dotenv(DRIVE_ROOT / '.env')
logger = setup_logging()
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

In [2]:
model = YOLO('yolo11m.pt')

In [3]:
results = model.train(
    data=str(CONFIGS_DIR / 'dataset.yaml'),
    epochs=100,
    batch=16,
    imgsz=640,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    warmup_epochs=3,
    cos_lr=True,
    device=DEVICE,
    project=str(CHECKPOINTS / 'yolov11'),
    name='run',
    save_period=10,
    exist_ok=True,
    plots=True,
)
print('Training complete.')
print('Best mAP@50:', results.results_dict.get('metrics/mAP50(B)'))

Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/data_prepared/configs/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=run, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_m

In [4]:
import json

summary = {
    'model': 'yolov11',
    'best_map50': results.results_dict.get('metrics/mAP50(B)'),
    'best_map50_95': results.results_dict.get('metrics/mAP50-95(B)'),
    'epochs': 100,
}
out = CHECKPOINTS / 'yolov11' / 'training_summary.json'
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(summary, indent=2))
print('Summary saved to', out)

Summary saved to /content/drive/MyDrive/FON/master_rad/checkpoints/yolov11/training_summary.json
